# 13 · Keras e TensorFlow: ler, manter e traduzir

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados · *notebook complementar*

Você aprendeu redes neurais com o PyTorch. Uma parte grande do código que existe em empresas, tutoriais e artigos, porém, foi escrita em **Keras** e **TensorFlow**. Esta aula é uma aula de leitura: vamos refazer a rede da aula 08 em Keras, abrir o capô do TensorFlow o suficiente para entender código de terceiros, e montar um dicionário de tradução entre os dois mundos.

## O que você será capaz de fazer ao final

1. Situar TensorFlow, Keras, PyTorch e JAX, e dizer onde cada um aparece.
2. Definir, treinar e avaliar um modelo em Keras, e relacionar cada passo com o que você escreveu à mão em PyTorch.
3. Reconhecer as três formas de definir um modelo em Keras.
4. Ler um laço de treino com `tf.data` e `tf.GradientTape`.
5. Executar o mesmo código Keras sobre motores diferentes, e salvar e carregar modelos.
6. Reconhecer código legado e traduzir entre Keras e PyTorch, incluindo a diferença na ordem dos eixos das imagens.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"       # o motor do Keras é escolhido ANTES de importá-lo
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"         # reduz as mensagens informativas do TensorFlow

import gzip
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlretrieve

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers

SEMENTE = 42
print("keras", keras.__version__, "· tensorflow", tf.__version__, "· motor do Keras:", keras.backend.backend(), "· GPUs:", len(tf.config.list_physical_devices("GPU")))
assert keras.backend.backend() == "tensorflow", "O Keras já tinha sido importado com outro motor: reinicie a sessão e execute de novo."

AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6b6b6b"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11})

## 1. O mapa dos frameworks

| | O que é | Onde aparece |
|---|---|---|
| **TensorFlow** | biblioteca de tensores e diferenciação automática do Google (2015) | sistemas em produção há anos; dispositivos móveis e embarcados (LiteRT, antigo TensorFlow Lite); código de artigos até por volta de 2020 |
| **Keras** | interface de alto nível para definir e treinar redes; nasceu independente, viveu dentro do TensorFlow (`tf.keras`) e, desde a versão 3, voltou a ser independente e funciona sobre **vários motores** | tutoriais, cursos, prototipagem, muito código de empresa |
| **PyTorch** | biblioteca de tensores e diferenciação automática da Meta (2016) | a grande maioria da pesquisa e dos modelos publicados hoje; é o que este curso usa |
| **JAX** | diferenciação automática e compilação sobre a interface do NumPy (Google) | pesquisa de alto desempenho |

No anúncio da versão 2.21, em março de 2026, a própria equipe do TensorFlow disse que o foco passa a ser correções de segurança e de erros críticos, e recomendou Keras 3, JAX e PyTorch para trabalho novo em IA generativa (o texto está em `developers.googleblog.com/en/whats-new-in-tensorflow-221/`). Na prática:

- **não comece um projeto novo em TensorFlow puro**;
- **Keras 3** continua em desenvolvimento ativo, e roda sobre TensorFlow, PyTorch ou JAX;
- o TensorFlow continua existindo em produção, e por isso **ler** TensorFlow continua sendo uma habilidade útil.

## 2. A rede da aula 08, em Keras

Os dados são os mesmos da aula 08: Fashion-MNIST, lido dos arquivos originais.

In [ ]:
BASE = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion/"
PASTA = Path("fashion_mnist")
PASTA.mkdir(exist_ok=True)
for arquivo in ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz", "t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"]:
    if not (PASTA / arquivo).exists():
        urlretrieve(BASE + arquivo, PASTA / arquivo)


def ler_imagens(caminho):
    with gzip.open(caminho) as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=16).reshape(-1, 28, 28)


def ler_rotulos(caminho):
    with gzip.open(caminho) as f:
        return np.frombuffer(f.read(), dtype=np.uint8, offset=8)


imagens, rotulos = ler_imagens(PASTA / "train-images-idx3-ubyte.gz"), ler_rotulos(PASTA / "train-labels-idx1-ubyte.gz")
imagens_teste, rotulos_teste = ler_imagens(PASTA / "t10k-images-idx3-ubyte.gz"), ler_rotulos(PASTA / "t10k-labels-idx1-ubyte.gz")
NOMES = ["Camiseta", "Calça", "Suéter", "Vestido", "Casaco", "Sandália", "Camisa", "Tênis", "Bolsa", "Bota"]

ordem = np.random.default_rng(SEMENTE).permutation(60_000)
ind_ajuste, ind_valid = ordem[:50_000], ordem[50_000:]
MEDIA, DESVIO = float(imagens[ind_ajuste].mean() / 255), float(imagens[ind_ajuste].std() / 255)


def preparar(lote_uint8):
    """uint8 (N, 28, 28) -> float32 (N, 28, 28, 1), padronizado. Repare na posição do eixo do canal."""
    return (((lote_uint8.astype("float32") / 255) - MEDIA) / DESVIO).astype("float32")[..., np.newaxis]


x_ajuste, y_ajuste = preparar(imagens[ind_ajuste]), rotulos[ind_ajuste].astype("int64")
x_valid, y_valid = preparar(imagens[ind_valid]), rotulos[ind_valid].astype("int64")
x_teste, y_teste = preparar(imagens_teste), rotulos_teste.astype("int64")
print("ajuste:", x_ajuste.shape, x_ajuste.dtype, "· validação:", x_valid.shape, "· teste:", x_teste.shape)

> ⚠️ **A primeira diferença, e a que mais causa erros na tradução:** a ordem dos eixos de uma imagem.
>
> | | Ordem | Um lote de Fashion-MNIST |
> |---|---|---|
> | PyTorch | lote, **canais**, altura, largura (*channels first*) | `(N, 1, 28, 28)` |
> | Keras e TensorFlow | lote, altura, largura, **canais** (*channels last*) | `(N, 28, 28, 1)` |

### Definir

A mesma arquitetura da `RedeConvolucional` da aula 08: duas convoluções com *max pooling*, uma camada densa de 64 neurônios e 10 saídas.

In [ ]:
keras.utils.set_random_seed(SEMENTE)              # fixa as sementes do Python, do NumPy e do motor

rede = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),                                   # a forma de UM exemplo, sem o eixo do lote
    layers.Conv2D(16, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10),                                                 # 10 logits, sem ativação
])
rede.summary()

Três diferenças em relação ao PyTorch já aparecem:

- **Não se informa o número de entradas de cada camada.** O Keras deduz as formas a partir do `Input`; a conta `32 * 7 * 7`, que fizemos à mão na aula 08, some.
- **A ativação é um argumento da camada** (`activation="relu"`), em vez de uma camada separada.
- **`summary()`** mostra as formas e os parâmetros de cada camada. O total, 105.866, é exatamente o da rede em PyTorch.

### Compilar e treinar

Em Keras, o laço de treino vem pronto. Dizemos **o que** usar em `compile`, e `fit` executa.

In [ ]:
rede.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),      # rótulos inteiros; a rede devolve logits
    metrics=["accuracy"],
)

freios = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),      # parada antecipada
    keras.callbacks.ModelCheckpoint("melhor_rede.keras", monitor="val_loss", save_best_only=True),  # checkpoint do melhor
]

inicio = time.perf_counter()
historico = rede.fit(
    x_ajuste, y_ajuste,
    validation_data=(x_valid, y_valid),
    epochs=6, batch_size=128,
    callbacks=freios,
    verbose=2,                                    # uma linha por época
)
print(f"\n{time.perf_counter() - inicio:.0f} s")

In [ ]:
curvas = pd.DataFrame(historico.history)          # o histórico é um dicionário de listas, uma por métrica
curvas.index = curvas.index + 1

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(curvas.index, curvas["loss"], color=CINZA, ls="--", label="ajuste")
ax.plot(curvas.index, curvas["val_loss"], color=AZUL, label="validação")
ax.set_xlabel("época"); ax.set_ylabel("perda"); ax.legend(frameon=False); ax.set_title("O mesmo treino da aula 08, sem escrever o laço")

perda_valid, acuracia_valid = rede.evaluate(x_valid, y_valid, verbose=0)
print(f"validação: perda {perda_valid:.4f} · acurácia {acuracia_valid:.1%}")
curvas.round(4)

O que você escreveu à mão na aula 08, e o que o Keras chama cada coisa:

| Aula 08 (PyTorch) | Keras |
|---|---|
| `Dataset` e `DataLoader` com `batch_size` e `shuffle` | arrays entregues ao `fit`, com `batch_size` (embaralha por padrão) |
| função de perda e otimizador criados no código | `compile(loss=..., optimizer=...)` |
| laço de épocas e de lotes, com os cinco passos | `fit(...)` |
| função `avaliar` | `evaluate(...)` |
| previsões com `no_grad` e `eval()` | `predict(...)` |
| parada antecipada escrita na função `ajustar` | `callbacks.EarlyStopping` |
| `torch.save` quando a validação melhora | `callbacks.ModelCheckpoint(save_best_only=True)` |
| recarregar o melhor estado ao fim | `restore_best_weights=True` |
| `modelo.train()` e `modelo.eval()` | automático: `fit` treina, `evaluate` e `predict` avaliam |
| `nn.CrossEntropyLoss()` sobre *logits* | `SparseCategoricalCrossentropy(from_logits=True)` |
| mover modelo e lotes para a GPU | automático |

O Keras troca **controle** por **concisão**. Para um treino padrão, são dez linhas contra oitenta. Para qualquer coisa fora do padrão, é preciso descer um nível (seção 4), e aí o código fica parecido com o do PyTorch.

> ⚠️ **`from_logits=True` é o equivalente Keras do erro da aula 07.** Sem esse argumento, a perda supõe que a rede já devolve probabilidades. Muita gente termina a rede com `Dense(10, activation="softmax")` e usa a perda sem `from_logits`: também funciona, e é o estilo mais comum em tutoriais. O erro é misturar: *softmax* na rede **e** `from_logits=True`, ou nenhum dos dois.

### ✏️ Exercício 1 (★★): um MLP em Keras

Crie `mlp`, um `keras.Sequential` equivalente ao MLP da aula 08: entrada `(28, 28, 1)`, `Flatten`, camada densa de 128 neurônios com ReLU e camada de saída com 10 *logits*. Compile como a rede acima e treine por 3 épocas, com lotes de 128 e validação, guardando o resultado do `fit` em `historico_ex1`. Use `verbose=2`.

In [ ]:
# ✏️ Exercício 1
keras.utils.set_random_seed(SEMENTE)
# SEU CÓDIGO AQUI
mlp = ...
historico_ex1 = ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
assert mlp.count_params() == 784 * 128 + 128 + 128 * 10 + 10, "o número de parâmetros não corresponde a 784 -> 128 -> 10"
assert mlp.output_shape == (None, 10) and len(historico_ex1.history["val_accuracy"]) == 3
_saida = mlp.predict(x_valid[:200], verbose=0)
assert not np.allclose(_saida.sum(axis=1), 1, atol=1e-3), "a última camada deve devolver logits, sem softmax"
assert historico_ex1.history["val_accuracy"][-1] > 0.8, "a acurácia ficou baixa: a perda foi criada com from_logits=True?"
print(f"Exercício 1 correto: {mlp.count_params()} parâmetros (os mesmos do MLP em PyTorch) · acurácia de validação {historico_ex1.history['val_accuracy'][-1]:.1%}")

## 3. Três formas de definir um modelo

Ao ler código Keras, você vai encontrar três estilos.

| Estilo | Como é | Quando aparece |
|---|---|---|
| **`Sequential`** | lista de camadas | modelos em fila, como o nosso |
| **API funcional** | cada camada é **chamada** sobre a saída da anterior; no fim, `keras.Model(entradas, saidas)` | modelos com várias entradas ou saídas, ramificações, conexões de atalho |
| **Subclasse de `keras.Model`** | camadas no `__init__`, conta no `call` | lógica própria no cálculo; é o estilo que se parece com `nn.Module` |

In [ ]:
# API funcional: a mesma rede, descrita como um grafo de camadas
entrada = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(16, 3, padding="same", activation="relu")(entrada)       # camada(tensor) devolve um tensor simbólico
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation="relu")(x)
saida = layers.Dense(10)(x)
rede_funcional = keras.Model(entrada, saida)
print("API funcional:", rede_funcional.count_params(), "parâmetros")


# Subclasse: o estilo do nn.Module
class RedeEmClasse(keras.Model):
    def __init__(self):
        super().__init__()
        self.extrator = keras.Sequential([layers.Conv2D(16, 3, padding="same", activation="relu"), layers.MaxPooling2D(2),
                                          layers.Conv2D(32, 3, padding="same", activation="relu"), layers.MaxPooling2D(2)])
        self.classificador = keras.Sequential([layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dense(10)])

    def call(self, x, training=False):                     # `call` faz o papel do `forward`
        return self.classificador(self.extrator(x, training=training), training=training)


rede_em_classe = RedeEmClasse()
print("subclasse:    ", tuple(rede_em_classe(x_valid[:4]).shape), "→", rede_em_classe.count_params(), "parâmetros")

Na subclasse aparece o argumento **`training`**. É a versão Keras de `modelo.train()` e `modelo.eval()`: em vez de um estado guardado no modelo, o modo é passado **a cada chamada**. Camadas como *dropout* e normalização em lote o consultam; `fit` passa `training=True`, e `evaluate` e `predict` passam `False`.

### ✏️ Exercício 2 (★★): do PyTorch para o Keras

Traduza para Keras o módulo abaixo, no estilo que preferir (funcional ou subclasse), e guarde o modelo em `rede_ex2`. A entrada do modelo Keras é um vetor de 784 números.

```python
class RedePequena(nn.Module):                 # PyTorch
    def __init__(self):
        super().__init__()
        self.oculta = nn.Linear(784, 64)
        self.dropout = nn.Dropout(0.2)
        self.saida = nn.Linear(64, 10)

    def forward(self, x):
        return self.saida(self.dropout(torch.relu(self.oculta(x))))
```

In [ ]:
# ✏️ Exercício 2
# SEU CÓDIGO AQUI
rede_ex2 = ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
_lote = np.random.default_rng(0).normal(size=(64, 784)).astype("float32")
_a, _b = np.asarray(rede_ex2(_lote, training=False)), np.asarray(rede_ex2(_lote, training=False))
_treino = np.asarray(rede_ex2(_lote, training=True))
assert _a.shape == (64, 10) and rede_ex2.count_params() == 784 * 64 + 64 + 64 * 10 + 10, "confira as camadas: 784 -> 64 -> 10"
assert np.allclose(_a, _b), "em modo de avaliação, duas chamadas devem dar o mesmo resultado"
assert not np.allclose(_a, _treino), "em modo de treino, o dropout deveria alterar a saída: ele está no modelo?"
assert not np.allclose(_a.sum(axis=1), 1, atol=1e-3), "a saída deve ser de logits, como no módulo original"
print("Exercício 2 correto: mesmas camadas, mesmos parâmetros, dropout ativo só com training=True.")

## 4. Por baixo do `fit`

Quando o treino sai do padrão (duas redes treinadas juntas, perdas compostas, pesquisa), o código Keras/TensorFlow desce para o laço explícito. É o trecho que mais se parece com o PyTorch, e o que mais aparece em repositórios de artigos.

### `tf.data`: o `DataLoader` do TensorFlow

In [ ]:
conjunto_ajuste = (
    tf.data.Dataset.from_tensor_slices((x_ajuste[:10_000], y_ajuste[:10_000]))      # um exemplo por elemento
    .shuffle(10_000, seed=SEMENTE)                                                # embaralha
    .batch(128)                                                                   # agrupa em lotes
    .prefetch(tf.data.AUTOTUNE)                                                   # prepara o próximo lote enquanto o atual é usado
)
conjunto_valid = tf.data.Dataset.from_tensor_slices((x_valid, y_valid)).batch(1000)

for xb, yb in conjunto_ajuste.take(1):
    print("um lote:", xb.shape, xb.dtype, "·", yb.shape, yb.dtype, "· tipo:", type(xb).__name__)

Um `tf.data.Dataset` é uma cadeia de transformações, no espírito do encadeamento do pandas e do Polars. Os lotes são tensores do TensorFlow (`EagerTensor`), que se convertem em NumPy com `.numpy()`.

### `tf.GradientTape`: o *autograd* do TensorFlow

No PyTorch, todo tensor com `requires_grad=True` registra as contas. No TensorFlow, o registro só acontece **dentro de um bloco** `with tf.GradientTape()`.

In [ ]:
x = tf.Variable(2.0)
with tf.GradientTape() as fita:
    y = x ** 2 + 3 * x                    # a mesma conta da aula 07
print("dy/dx =", float(fita.gradient(y, x)), "(a aula 07 chegou a 7 com x.grad)")

In [ ]:
keras.utils.set_random_seed(SEMENTE)
rede_manual = keras.models.clone_model(rede)                      # mesma arquitetura, pesos novos
funcao_perda = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
otimizador = keras.optimizers.Adam(1e-3)


@tf.function                                                       # compila a função em um grafo: bem mais rápido
def passo_de_treino(xb, yb):
    with tf.GradientTape() as fita:
        logits = rede_manual(xb, training=True)                    # prever
        perda = funcao_perda(yb, logits)                           # medir a perda (atenção: rótulos primeiro)
    gradientes = fita.gradient(perda, rede_manual.trainable_variables)                       # retropropagar
    otimizador.apply_gradients(zip(gradientes, rede_manual.trainable_variables))             # atualizar
    return perda


inicio = time.perf_counter()
for epoca in range(1, 3):
    perdas = [float(passo_de_treino(xb, yb)) for xb, yb in conjunto_ajuste]
    print(f"época {epoca} · perda média de ajuste {np.mean(perdas):.4f}")
print(f"{time.perf_counter() - inicio:.1f} s para 2 épocas em 10 mil imagens")

| Os cinco passos da aula 07 | PyTorch | TensorFlow |
|---|---|---|
| 1. zerar | `otimizador.zero_grad()` | não existe: cada `GradientTape` começa do zero |
| 2. prever | `logits = modelo(xb)` | `logits = modelo(xb, training=True)`, dentro da fita |
| 3. medir a perda | `perda = funcao_perda(logits, yb)` | `perda = funcao_perda(yb, logits)` (**ordem invertida**) |
| 4. retropropagar | `perda.backward()` | `gradientes = fita.gradient(perda, variaveis)` |
| 5. atualizar | `otimizador.step()` | `otimizador.apply_gradients(zip(gradientes, variaveis))` |

Dois detalhes derrubam quem traduz de um para o outro: a **ordem dos argumentos da perda** (rótulos primeiro no Keras, previsões primeiro no PyTorch) e o fato de os gradientes serem **devolvidos** pela fita, em vez de ficarem guardados nos parâmetros.

**`@tf.function`** transforma a função Python em um grafo de operações do TensorFlow, executado sem passar pelo interpretador. A armadilha: o código Python da função roda **uma vez**, na compilação. Um `print` lá dentro aparece só na primeira chamada, e condições que dependem de valores de tensores precisam de construções do próprio TensorFlow.

### ✏️ Exercício 3 (★★): um laço de avaliação

Escreva `acuracia_tf(modelo, conjunto)`: percorre um `tf.data.Dataset` de lotes `(imagens, rótulos)`, chama o modelo em modo de avaliação e devolve a proporção de acertos, como um `float` do Python. A classe prevista é o `argmax` dos *logits* (`tf.argmax(logits, axis=1)`, ou converta para NumPy).

In [ ]:
# ✏️ Exercício 3
def acuracia_tf(modelo, conjunto):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
_minha = acuracia_tf(rede, conjunto_valid)
_do_keras = rede.evaluate(x_valid, y_valid, verbose=0)[1]
assert isinstance(_minha, float) and abs(_minha - _do_keras) < 1e-4, f"a sua acurácia ({_minha:.4f}) difere da calculada pelo Keras ({_do_keras:.4f})"
print(f"Exercício 3 correto: {_minha:.2%}, igual ao evaluate do Keras. A rede do laço manual, com 2 épocas em 10 mil imagens: {acuracia_tf(rede_manual, conjunto_valid):.2%}")

## 5. Um código, vários motores

O Keras 3 separa a **descrição** do modelo do **motor** que faz as contas. O mesmo arquivo roda sobre TensorFlow, PyTorch ou JAX; basta mudar a variável de ambiente `KERAS_BACKEND` antes de importar o Keras.

Como o motor só pode ser escolhido uma vez por sessão, vamos gravar um pequeno programa em arquivo e executá-lo duas vezes, em processos separados.

In [ ]:
PROGRAMA = '''
import numpy as np
import keras
from keras import layers

keras.utils.set_random_seed(0)
dados = np.load("amostra_fashion.npz")
rede = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, 3, padding="same", activation="relu"), layers.MaxPooling2D(2),
    layers.Flatten(), layers.Dense(10),
])
rede.compile(optimizer="adam", loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
rede.fit(dados["x"], dados["y"], epochs=2, batch_size=128, verbose=0)
print(f"motor: {keras.backend.backend():<11} acurácia de validação: {rede.evaluate(dados['xv'], dados['yv'], verbose=0)[1]:.3f}")
'''
Path("treino_keras.py").write_text(PROGRAMA, encoding="utf-8")
np.savez("amostra_fashion.npz", x=x_ajuste[:10_000], y=y_ajuste[:10_000], xv=x_valid[:2000], yv=y_valid[:2000])

for motor in ["tensorflow", "torch"]:
    resultado = subprocess.run([sys.executable, "treino_keras.py"], capture_output=True, text=True,
                               env={**os.environ, "KERAS_BACKEND": motor, "TF_CPP_MIN_LOG_LEVEL": "3"})
    print(resultado.stdout.strip() or f"motor {motor}: falhou → {resultado.stderr.strip().splitlines()[-1][:200]}")

O mesmo arquivo, sem uma linha alterada, treinou sobre dois motores. Os números diferem um pouco porque cada motor sorteia os pesos iniciais à sua maneira.

Na prática, isso significa que um modelo Keras pode ser treinado com o motor que a equipe preferir e convive, no mesmo projeto, com código PyTorch: com o motor `torch`, um modelo Keras **é** um `torch.nn.Module`. Para escrever contas próprias que funcionem em qualquer motor, usa-se `keras.ops` (por exemplo `keras.ops.mean`), em vez de funções do TensorFlow ou do PyTorch.

## 6. Salvar e carregar

| Formato | Como | Contém | Uso |
|---|---|---|---|
| **`.keras`** | `modelo.save("m.keras")` | arquitetura, pesos e configuração de treino, em um arquivo | o padrão atual |
| `.weights.h5` | `modelo.save_weights(...)` | só os pesos | quando a arquitetura está no código, como o `state_dict` |
| `.h5` | `modelo.save("m.h5")` | formato antigo do Keras 2 | só para ler modelos legados |
| *SavedModel* | `modelo.export("pasta")` | grafo para servir | TF Serving, LiteRT; não serve para continuar o treino |

In [ ]:
rede.save("rede_fashion.keras")
recarregada = keras.saving.load_model("rede_fashion.keras")

iguais = np.allclose(np.asarray(rede(x_valid[:100], training=False)), np.asarray(recarregada(x_valid[:100], training=False)), atol=1e-5)
print(f"rede_fashion.keras: {Path('rede_fashion.keras').stat().st_size / 1e6:.1f} MB · previsões idênticas depois de recarregar: {iguais}")

Diferente do `state_dict` do PyTorch, o arquivo `.keras` guarda **a arquitetura junto**: para carregar, não é preciso ter o código que definiu o modelo.

> ⚠️ Como em qualquer formato de modelo, só carregue arquivos de origem confiável. O formato `.keras` tem um modo seguro, ligado por padrão, que recusa código arbitrário embutido; os formatos antigos (`.h5`, *pickle*) não têm essa proteção.

## 7. Código legado: como reconhecer

Você vai encontrar três gerações de código. Saber a qual delas um arquivo pertence diz quanto trabalho ele dará.

| Geração | Sinais no código | O que fazer |
|---|---|---|
| **TensorFlow 1** (até 2019) | `tf.Session()`, `sess.run(...)`, `tf.placeholder`, `feed_dict`, `tf.contrib`, `tf.compat.v1` | não roda nas versões atuais sem adaptação; reescrever costuma ser mais barato que migrar |
| **`tf.keras` / Keras 2** (2019 a 2023) | `from tensorflow import keras`, `from tensorflow.keras.layers import ...`, `model.fit_generator`, `Adam(lr=...)`, arquivos `.h5` | em geral basta trocar os *imports* para `import keras` e atualizar poucos nomes |
| **Keras 3** (2023 em diante) | `import keras`, `keras.ops`, `KERAS_BACKEND`, arquivos `.keras` | é o estilo desta aula |

Um trecho típico da primeira geração, para reconhecimento (não execute):

```python
x = tf.placeholder(tf.float32, [None, 784])            # "lugar reservado" para os dados
W = tf.Variable(tf.zeros([784, 10]))
y = tf.nn.softmax(tf.matmul(x, W))
perda = tf.reduce_mean(-tf.reduce_sum(y_real * tf.log(y), axis=1))
treinar = tf.train.GradientDescentOptimizer(0.5).minimize(perda)

with tf.Session() as sess:                              # nada roda fora de uma sessão
    sess.run(tf.global_variables_initializer())
    for _ in range(1000):
        sess.run(treinar, feed_dict={x: lote_x, y_real: lote_y})
```

Nesse estilo, o código primeiro **descreve** um grafo de operações e só depois o executa dentro de uma sessão. Foi a dificuldade de depurar esse modelo que abriu espaço para o PyTorch, em que cada linha executa na hora.

## 8. Dicionário de tradução

| Conceito | PyTorch | Keras 3 |
|---|---|---|
| Camada densa | `nn.Linear(entradas, saidas)` | `layers.Dense(saidas)` |
| Convolução | `nn.Conv2d(c_ent, c_sai, 3, padding=1)` | `layers.Conv2D(c_sai, 3, padding="same")` |
| *Pooling* | `nn.MaxPool2d(2)` | `layers.MaxPooling2D(2)` |
| Achatar | `nn.Flatten()` | `layers.Flatten()` |
| *Dropout* | `nn.Dropout(0.3)` | `layers.Dropout(0.3)` |
| Normalização em lote | `nn.BatchNorm2d(c)` | `layers.BatchNormalization()` |
| Ativação | `nn.ReLU()` ou `torch.relu` | `activation="relu"` ou `layers.ReLU()` |
| Fila de camadas | `nn.Sequential(...)` | `keras.Sequential([...])` |
| Modelo próprio | `nn.Module`: `__init__` e `forward` | `keras.Model`: `__init__` e `call` |
| Modo de treino | `modelo.train()` / `modelo.eval()` | argumento `training=True/False` |
| Perda multiclasse | `nn.CrossEntropyLoss()(logits, y)` | `SparseCategoricalCrossentropy(from_logits=True)(y, logits)` |
| Perda binária | `nn.BCEWithLogitsLoss()` | `BinaryCrossentropy(from_logits=True)` |
| Otimizador | `torch.optim.AdamW(params, lr=...)` | `keras.optimizers.AdamW(learning_rate=...)` |
| Imagens | `(N, C, A, L)` | `(N, A, L, C)` |
| Pesos | `state_dict()` | `get_weights()`, arquivo `.keras` |
| Número de parâmetros | `sum(p.numel() for p in m.parameters())` | `m.count_params()` |

### ✏️ Exercício 4 (★): a ordem dos eixos

Escreva as duas funções de conversão de lotes de imagens entre as convenções:

- `para_keras(lote)`: de `(N, C, A, L)` para `(N, A, L, C)`;
- `para_torch(lote)`: o caminho de volta.

Use `np.transpose`. Atenção: `reshape` **não** serve; ele reorganizaria os pixels sem trocar os eixos.

In [ ]:
# ✏️ Exercício 4
def para_keras(lote):
    # SEU CÓDIGO AQUI
    ...


def para_torch(lote):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
_colorido = np.random.default_rng(0).integers(0, 255, size=(5, 3, 32, 48))            # estilo PyTorch: 5 imagens, 3 canais, 32 × 48
_k = para_keras(_colorido)
assert _k.shape == (5, 32, 48, 3), "a forma no estilo Keras deveria ser (5, 32, 48, 3)"
assert (_k[2, 10, 20, :] == _colorido[2, :, 10, 20]).all(), "o pixel (10, 20) da imagem 2 mudou: foi usado reshape?"
assert (para_torch(_k) == _colorido).all(), "ir e voltar deve devolver o lote original"
assert para_torch(x_valid[:8]).shape == (8, 1, 28, 28)
print("Exercício 4 correto: os eixos trocam de lugar e cada pixel continua sendo o mesmo.")

### ✏️ Exercício 5 (★★): revisão de código

Um colega encontrou este trecho em um repositório de 2021 e quer usá-lo hoje, com Keras 3. Aponte **quatro** coisas que precisam mudar ou que estão erradas, e diga como corrigir cada uma.

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten
from tensorflow.keras.optimizers import Adam

model = Sequential()
model.add(Conv2D(32, 3, activation="relu", input_shape=(1, 28, 28)))
model.add(Flatten())
model.add(Dense(10, activation="softmax"))
model.compile(optimizer=Adam(lr=0.001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
model.fit_generator(gerador_de_lotes, epochs=10)
model.save("modelo.h5")
```

✏️ **Sua resposta:**

_Escreva aqui._

## Para levar

### Quando usar o quê

| Situação | Escolha razoável |
|---|---|
| Projeto novo de aprendizado profundo | PyTorch |
| Protótipo rápido de um modelo padrão; equipe que já usa Keras | Keras 3, com o motor que a equipe preferir |
| Manter um sistema existente em TensorFlow | continuar nele, planejando a migração quando houver mudança grande |
| Usar modelos pré-treinados de texto, imagem ou áudio | PyTorch, com o Hugging Face (aula 09) |
| Rodar um modelo em celular ou microcontrolador | exportar para o formato do dispositivo (LiteRT, ONNX, ExecuTorch) |
| Levar um modelo de um framework para outro | ONNX como formato de troca, ou reescrever a arquitetura e copiar os pesos |

### Erros comuns

| Erro | Consequência | Como evitar |
|---|---|---|
| Imagens na ordem do PyTorch em um modelo Keras (ou o contrário) | erro de forma, ou treino ruim sem erro | conferir a forma; `np.transpose` |
| *softmax* na rede e `from_logits=True` (ou nenhum dos dois) | o modelo aprende mal | escolher uma das duas convenções |
| Trocar a ordem dos argumentos da perda ao traduzir | perda sem sentido | Keras: `(y, previsão)`; PyTorch: `(previsão, y)` |
| Definir `KERAS_BACKEND` depois de importar o Keras | a variável é ignorada | definir na primeira célula; reiniciar a sessão |
| `print` ou lógica Python dentro de `@tf.function` | executa uma vez só | `tf.print`, ou retirar o decorador para depurar |
| Esquecer `training=True` no laço manual | *dropout* e normalização em modo de avaliação durante o treino | passar o argumento explicitamente |
| Misturar `tf.keras` e `keras` no mesmo projeto | objetos incompatíveis entre si | um único estilo de importação |
| Carregar `.h5` ou *pickle* de terceiros | execução de código arbitrário | preferir `.keras`, de origem confiável |

### Resumo

- O TensorFlow está em modo de manutenção para trabalho novo; o **Keras 3** segue ativo e roda sobre TensorFlow, PyTorch ou JAX.
- Em Keras, `compile` e `fit` substituem o laço de treino; **callbacks** fazem a parada antecipada e o *checkpoint*.
- Três estilos de definição: `Sequential`, API funcional e subclasse (`call` no lugar de `forward`).
- Por baixo do `fit`: `tf.data` monta os lotes e `tf.GradientTape` registra as contas e devolve os gradientes.
- Ao traduzir, os três pontos de atenção são a **ordem dos eixos**, a convenção de ***logits*** e a **ordem dos argumentos da perda**.
- Código com `tf.Session` e `placeholder` é da primeira geração: reconheça, e avalie reescrever.

### Para casa

A lista de exercícios está em `13_casa.ipynb`.